In [ ]:
# Cell 1: Setup & Initialization
import os
import sys
import torch
import matplotlib.pyplot as plt
from torch.utils.data import random_split

sys.path.append(os.path.abspath(".."))
import config
from src.preprocessing.dataset import get_dataloader, ECGSpikeDataset
from src.models.rmlp_snn import RecurrentMLPSNN
from src.training.trainer import train_epoch, evaluate
from src.training.loss import MultiObjectiveSNNLoss

# Cell 2: Dataset Ingestion & Splits
full_dataset = ECGSpikeDataset(data_dir=config.DATA_RAW_DIR, delta_v=config.DELTA_V)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_set, val_set = random_split(full_dataset, [train_size, val_size])
train_loader = torch.utils.data.DataLoader(train_set, batch_size=config.BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=config.BATCH_SIZE, shuffle=False)

print(f"Train Dataset Size: {train_size} | Validation Dataset Size: {val_size}")

# Cell 3: Model & Optimizer Initialization
model = RecurrentMLPSNN(input_dim=config.INPUT_DIM, beta=config.LIF_BETA).to(config.DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
criterion = MultiObjectiveSNNLoss(
    lambda_s=config.LAMBDA_SPARSITY, 
    lambda_w=config.LAMBDA_WEIGHT_DECAY,
    target_firing_rate=config.TARGET_FIRING_RATE
)

# Cell 4: Training & Curve Collection
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, config.EPOCHS + 1):
    t_loss, t_acc = train_epoch(model, train_loader, optimizer, criterion, config.DEVICE)
    v_loss, v_acc = evaluate(model, val_loader, criterion, config.DEVICE)
    
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)
    
    print(f"Epoch [{epoch}/{config.EPOCHS}] -> Train Acc: {t_acc:.2f}% | Val Acc: {v_acc:.2f}%")

# Save Baseline Trained Weights
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(config.CHECKPOINT_DIR, "baseline_rmlp_snn.pth"))

# Cell 5: Plot Training Loss & Accuracy Metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss', color='crimson')
ax1.plot(history['val_loss'], label='Val Loss', color='navy', linestyle='--')
ax1.set_title('Triple-Objective Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train Acc', color='seagreen')
ax2.plot(history['val_acc'], label='Val Acc', color='darkorange', linestyle='--')
ax2.set_title('Classification Accuracy (%)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.PLOTS_DIR, "training_curves.png"), dpi=300)
plt.show()